In [1]:
print("starting...")

starting...


In [2]:
# pip install psycopg2
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    dbname="stocks",
    user="postgres",
    password="password",
    host="localhost",
    port="5432"
)
stock_code = 'NVDA'
sql = """
WITH CTE AS (
    SELECT 
        *,
        -- 15 minute interval grouping
        FLOOR(EXTRACT(EPOCH FROM "datetime") / (420 * 60)) AS interval_group,
        LAST_VALUE("close") OVER (
            PARTITION BY FLOOR(EXTRACT(EPOCH FROM "datetime") / (420 * 60)) 
            ORDER BY "datetime"
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_close_in_interval
    FROM stock
    WHERE stock_code = '""" + stock_code + """' 
)
SELECT
    MIN("datetime") AS start_time,
    MAX("datetime") AS end_time,
    MIN("low") AS low,
    MAX("high") AS high,
    MAX("last_close_in_interval") AS close,
    SUM("volume") AS volume
FROM CTE
GROUP BY interval_group
ORDER BY start_time;
"""

df = pd.read_sql(sql, conn)
conn.close()

ModuleNotFoundError: No module named 'pandas'

In [3]:
import os
import json
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np
from collections import defaultdict

model_name = "gpt-4.1"

def collect_daily_sentiment_for_stock(stock_code, base_dir):
    """
    Collect and aggregate sentiment scores per day for a stock code.
    Returns a list of dicts: {"date": date, "score": avg_score, "count": n, "labels": [labels]}
    """
    daily_scores = defaultdict(list)
    daily_labels = defaultdict(list)
    for date_folder in os.listdir(base_dir):
        date_path = os.path.join(base_dir, date_folder)
        if not os.path.isdir(date_path):
            print(f"Skipping {date_path}, not a directory")
            continue
        stock_path = os.path.join(date_path, stock_code, "detail", "news.json")
        # print(f"Processing {stock_path}")
        if os.path.exists(stock_path):
            with open(stock_path, "r", encoding="utf-8") as f:
                news_items = json.load(f)
            for item in news_items:
                analysis = item.get("analysis", {})
                # print(f"Found analysis for {item.get('title', 'Unknown')}: {analysis}")
                sentiment = analysis.get(model_name)
                if sentiment and float(sentiment["value"]) > 0.0:
                    daily_scores[date_folder].append(float(sentiment["value"]))
                    daily_labels[date_folder].append(sentiment["sentiment"])
    # Aggregate per day
    result = []
    for date, scores in daily_scores.items():
        avg_score = np.mean(scores)
        result.append({
            "date": date,
            "score": avg_score,
            "count": len(scores),
            "labels": daily_labels[date]
        })
    return sorted(result, key=lambda x: x["date"])

def plot_daily_sentiment(daily_sentiment, stock_code):
    """
    Plot daily aggregated sentiment scores over time for a stock, with a smooth curve.
    """
    if not daily_sentiment:
        print("No sentiment data found for", stock_code)
        return

    dates = [datetime.strptime(x["date"], "%Y-%m-%d") for x in daily_sentiment]
    scores = [x["score"] for x in daily_sentiment]

    # Interpolate for smooth curve
    if len(dates) > 1:
        date_nums = np.array([d.toordinal() for d in dates])
        interp_dates = np.linspace(date_nums.min(), date_nums.max(), 300)
        interp_scores = np.interp(interp_dates, date_nums, scores)
        interp_dates_dt = [datetime.fromordinal(int(d)) for d in interp_dates]
    else:
        interp_dates_dt = dates
        interp_scores = scores

    plt.figure(figsize=(12, 5))
    plt.plot(interp_dates_dt, interp_scores, label='Smoothed Sentiment', color='tab:blue')
    plt.scatter(dates, scores, color='tab:orange', label='Daily Avg Sentiment')
    plt.ylim(0, 1)
    plt.title(f"Daily Sentiment Score (Smoothed) for {stock_code}")
    plt.xlabel("Date")
    plt.ylabel("Sentiment Score (0=Negative, 1=Positive)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Example usage:
# stock_code = "AMZN"
base_dir = "c:/git/daniel/stocks/stocks/scrapper/news"
daily_sentiment = collect_daily_sentiment_for_stock(stock_code, base_dir)
plot_daily_sentiment(daily_sentiment, stock_code)

import matplotlib.pyplot as plt

# Drop rows with missing 'close' or 'start_time' values
df_filtered = df.dropna(subset=['start_time', 'close'])

# Convert start_time to string for categorical X axis
df_filtered['start_time_str'] = df_filtered['start_time'].astype(str)

import matplotlib.pyplot as plt

# Prepare data
dates_sentiment = [datetime.strptime(x["date"], "%Y-%m-%d") for x in daily_sentiment]
scores = [x["score"] for x in daily_sentiment]

# For price, use the same date format as sentiment for alignment
df_filtered['start_time_dt'] = pd.to_datetime(df_filtered['start_time'])
dates_price = df_filtered['start_time_dt']
close_prices = df_filtered['close']

fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot close price
color = 'tab:orange'
ax1.set_xlabel('Date')
ax1.set_ylabel('Close Price', color=color)
ax1.plot(dates_price, close_prices, color=color, marker='o', label='Close Price')
ax1.tick_params(axis='y', labelcolor=color)
ax1.legend(loc='upper left')
plt.xticks(rotation=45)

# Create a second y-axis for sentiment
ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Sentiment Score', color=color)
ax2.plot(dates_sentiment, scores, color=color, marker='x', label='Sentiment Score')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0, 1)
ax2.legend(loc='upper right')

plt.title(f"{stock_code} Close Price and Daily Sentiment")
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'